# Setup

In **Google Colab**, run the setup cell below: it downloads the versioned repository, installs DaliScope, and enables custom widgets. No local Python or Conda installation is needed. Use a standard CPU runtime.

In **local Jupyter**, install DaliScope using the root README and select **Python (DaliScope)**. If you update packages in an existing local session, restart the kernel once.

Run cells in order. The analysis setup locates the repository and sets the working directory to `notebooks` for relative data paths. See the [Colab guide](https://github.com/Liisa-Holm-group/DaliScope/blob/v0.1.9/docs/colab.md) for saving outputs and restarting a cloud runtime.


In [ ]:
# In Colab this installs DaliScope automatically. Locally, follow the README first.
import sys
from pathlib import Path
import subprocess
import runpy

try:
    import google.colab
except ModuleNotFoundError as error:
    if error.name not in {"google", "google.colab"}:
        raise
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    DALISCOPE_RELEASE = "v0.1.9"
    colab_root = Path("/content/DaliScope")
    if not colab_root.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", DALISCOPE_RELEASE,
            "https://github.com/Liisa-Holm-group/DaliScope.git", str(colab_root),
        ], check=True)
    setup_path = colab_root / "scripts/colab_setup.py"
    if not setup_path.is_file():
        raise RuntimeError("This runtime contains an older checkout. Start a fresh runtime and run setup again.")
    setup = runpy.run_path(str(setup_path))
    setup["setup_colab"](colab_root, DALISCOPE_RELEASE)
else:
    print("Local Jupyter: using the installed DaliScope environment.")


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start in the DaliScope root or notebooks directory")
os.chdir(ROOT / "notebooks")
import daliscope
print("DaliScope", daliscope.__version__)

In [ ]:
import pandas as pd
from IPython.display import display

import daliscope
import daliscope.core.project
import daliscope.optics.histograms
import daliscope.analysis.data_preview
import daliscope.analysis.occupancy
import daliscope.analysis.structal_similarity # for cluster evaluation
import daliscope.widgets.pairwise_py3Dmol
import daliscope.optics.modules_sunburst
import daliscope.analysis.communities
import daliscope.analysis.pfam_stats
import daliscope.optics.py3Dmol_viewer
import daliscope.optics.domnet_viewer
import daliscope.analysis.CH_cluster

from daliscope.analysis.data_preview import show_heading

# A multidomain Query: from architecture to sequence signatures

In this worked example, we use a multidomain Query protein to explore how structural and sequence features can be traced from an individual protein into a broader population.

We begin by examining the Query's domain architecture and structural relationships, then use sequence-level analyses to identify conserved features within selected domains. These observations lead to a discrete sequence signature, which we use to search the selected Domain 5 population and determine how often the feature occurs among its structurally matched targets. Finally, we relate the observed motif patterns to the available Pfam annotations and inspect representative structures.

The example therefore follows a progression from **domain architecture and structural context, through sequence conservation, to pattern search within the selected Domain 5 population and annotation-guided interpretation**, illustrating how the different DaliScope analyses can be combined into a single investigation.


<hr style="border: 2px solid #ccc;">

# <span style="color: #2b5c8f;">Explore the Dali search results</span>

<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    Structural similarity at a glance
</div>

---

Once the raw Dali data are loaded, DaliScope presents the search results as a scatterplot of **Z-score versus query coverage**. Together, these provide a first view of the quality and extent of each structural match.

The two measures are generally correlated. The Dali Z-score summarizes structural similarity based on the correspondence of intramolecular distances: for a given number of aligned residues, a more precise structural match produces a higher Z-score.

The plot uses two complementary colour schemes:

* **Sequence identity** indicates how the structural matches relate to what could be found by sequence-based searches: from clear sequence homologs, through the *twilight zone* where structural alignment becomes particularly informative, to cases where structural similarity is difficult to detect from sequence alone.
* **Z-score regimes** provide a heuristic indication of the strength of the structural match. These regimes work well for typical cases, but their interpretation depends on the query structure: optimal thresholds can vary with protein size and shape.

As a first step, this view gives us a map of the structural neighbourhood of the Query and helps identify which hits warrant closer inspection.


In [ ]:
# Load data
# The data pack must be uploaded to the Notebook host. Default data files are in DaliScope/notebooks/data.
pack_path = 'data/ZN_full.tar.gz'

# 1. Initialize & Preview Top Hits
project, FULL_DF, FULL_VIEW = daliscope.analysis.data_preview.load_and_inspect_project(pack_path)

# 2. Query metadata
show_heading("Query information")
tmp = list(project._query_meta.iloc[0])
puu_domains = tmp[5]
print(f"""
Identifier: {tmp[0]}
Length: {tmp[1]}
Description: {tmp[4]}
PUU domains: {puu_domains}
""")

# 3. Plot Landscape
fig = daliscope.analysis.data_preview.plot_dali_hits_with_boundaries(
    FULL_DF, 
    daliscope.analysis.data_preview.build_and_register_subsets(project, FULL_DF, FULL_VIEW)
)

# 4. Project structural alignments onto Query positions
project.viz.plot_msa(FULL_VIEW, 0, project.query_length, plot_type='heatmap', data_col='dssp_pileup')

The plot shows a mix of sequence identity regimes: homologs generally have the highest Z-scores, followed by twilight hits. The large peak of low-identity hits may contain biologically interesting matches as well as noise. DaliScope automatically creates two subsets, `FILTERED_0` and `FILTERED_1`, by splitting the plot at valleys in the density distribution. These subsets sometimes, but not always, correspond to families or superfamilies.

The structural alignment shows multiple distinct blocks: the Query protein contains several domains, with different numbers of hits. Refined superimposition and annotation should therefore be examined for each domain separately.


## Structural Coherence and Occupancy-Based Selection

Dali alignments optimize intramolecular distance similarity rather than spatial coordinates. Consequently, distant C-alpha atoms at similar radial distances can inflate global RMSD values and disrupt rigid-body superimpositions.

To restore spatial coherence within specified domains, this example uses two steps:
1. **Domain clipping:** Exclude off-domain aligned positions, then recompute a uniform least-squares superimposition using the retained positions.
2. **Occupancy-based selection:** Score each target using a position-specific occupancy profile. This selects targets but does not reweight the fitted coordinates.

In the secondary-structure-colored alignment plot (blue alpha-helices, red beta-strands, green loops), rigid domains appear as distinct structural blocks. Using this visual reference, we define five contiguous domains in **one-based, inclusive PDB residues**: `1-149`, `150-214`, `840-920`, `921-1100`, and `1343-1722`. The registration cell converts these to zero-based, half-open clipping ranges.

Each domain yields a clipped view `dom_{i}` and an occupancy-selected subset `dom_{i}_FOLD`. This example uses a normalized occupancy-score cutoff of **0.5** (strictly greater than 0.5); it does not require 80% unweighted residue coverage. The subset inherits its clipped fit.


In [ ]:
# Prepare PDB structure string
pdb_string = daliscope.optics.py3Dmol_viewer.xyz_to_ca_pdb(project.query_ca_coords, project.query_sequence)

custom_domains = "1-149, 150-214, 840-920, 921-1100, 1343-1722"

# Initialize and render interactive viewer
viewer = daliscope.optics.domnet_viewer.DomNetViewer(
    pdb_text=pdb_string,
    domain_string=custom_domains if custom_domains else puu_domains,
    normal_radius=0.65,
    highlight_radius=0.60,
)

print("""
Domain Syntax example: "1-100, 101-300_400-500, 301-399" means 
    Domain 1: 1–100; 
    Domain 2 [discontiguous]: 101–300 + 400–500; 
    Domain 3: 301–399
    
IF the viewer throws a JavaScript error, refresh your browser (CTRL-R) and rerun this code cell. 
If the canvas is blank, re-run this cell

You can play around editing and visualizing domain boundaries, though this worked example will use 
the initial domain boundaries for downstream analyses.
""")

viewer.display()

In [ ]:
from daliscope.mechanics.domain_ranges import pdb_domain_string_to_clipping

# 2. Register clipped domain views. Recompute superimpositions and metrics; Z-score remains global.
daliscope.mechanics.metrics.register_multiple_domains(
    domain_string=pdb_domain_string_to_clipping(custom_domains), 
    project=project, 
    custom_name="dom"
)

# Process each registered domain view
view_list = []
for i in [1,2,3,4,5]: 
    view_list.append(f"dom_{i}")

for view_name in view_list:
    daliscope.analysis.occupancy.create_fold_subset(project, view_name, show_plot=True, cutoff=0.5)

show_heading("Which parts of the protein correspond to distinct structural folds?")
shared_matrix = daliscope.analysis.occupancy.get_fold_matrix(project, view_list)

# Render heatmap
daliscope.analysis.occupancy.plot_shared_matrix_heatmap(shared_matrix, figsize=(3,2))

In this target set, Domain 3 and Domain 4 matches tend to co-occur with Domain 5 matches. The matrix above shows the overlap for the current run.

Matches to Domains 1 and 2 also occur in many proteins lacking matches to the Query's other domains. Some targets share additional Query domains, so these regions are not exclusive to separate architectures.


# What do existing domain classifications tell us?
Here, we show the Pfam family and clan assignments of the domain-fold populations. The table links family and clan identifiers to InterPro documentation. Domain 5 matches are predominantly associated with the CUPIN clan. Domains 3 and 4 are distinct cysteine-rich regions that often co-occur with a CUPIN-associated domain in the same proteins; they should not themselves be described as CUPIN domains. Domains 1 and 2 have an SH3-like fold.


In [ ]:
# 1. Table view
for i in [1,2,3,4,5]:
    view_name = f"dom_{i}_FOLD"
    show_heading(view_name)
    daliscope.analysis.pfam_stats.pfam_counts(view_name, project, k=10, print_tables = True)

# Play with the structures
The Query structure is blue, the target is red. Thick lines and dashed lines show the raw Dali alignment, but the least-squares superimposition uses clipped aligned residues with uniform weights. 

Note that Dali performs a local structural alignment, finding matches between small domains embedded in large multidomain proteins.

In [ ]:
# Interactive py3Dmol viewer
daliscope.widgets.pairwise_py3Dmol.generate_superimposition_view(
    target_ix=77,
    df=project.views['dom_2_FOLD'],
    project=project,
    q_ca_coords=project.query_ca_coords,
    target_ca_coords=project.coords,
)

## How do domain architectures vary across the target population?

Domain architecture cartoons provide a compact overview of domain composition and organization across the target population.

* **Dashed red boxes** mark the boundaries of the structural match.
* **Empty boxes** represent regions without Pfam annotations.
* **`dom_3_FOLD` and `dom_4_FOLD`** contain matches to regions that often occur alongside CUPIN-associated domains, while the matched regions themselves generally lack CUPIN Pfam assignments.
* **Partially occupied boxes** in `dom_5_FOLD` indicate that the structurally conserved core extends beyond Pfam's sequence profile model.


In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_1_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_2_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_3_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_4_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_5_FOLD', renderer='matplotlib', pfam_col='clan_domains')

## Where do the domain populations fall in the scatterplot?

The five domains preserve the original Z-scores calculated from the full-length protein comparisons. As a result, hits matching smaller individual domains cluster at lower Z-scores compared with those spanning larger or multiple domains. CUPIN clan (CL0029) annotations are most prominent in the `dom_5_FOLD` population. The `dom_3_FOLD` and `dom_4_FOLD` views capture other Query regions that frequently occur in proteins also containing a CUPIN-associated domain.


In [ ]:
# 3. Scatterplot colored by Pfam categories. Pool more distant Pfam categories as 'Other'
def _scatterplot(view_name):
    show_heading(view_name)
    df = project.views[view_name][['alignment_length', 'z_score', 'clan', 'pfam']].copy()
    df["pfam_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "pfam", k=10)
    df["clan_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "clan", k=10)
    display(daliscope.optics.histograms.plot_scatter(df, x='z_score', y='alignment_length', color='clan_label', engine='matplot',))

for i in [1,2,3,4,5]: 
    view_name = f'dom_{i}_FOLD'
    _scatterplot(view_name)

In [ ]:
project.views['dom_5_FOLD'].value_counts('clan')
len(project.views['dom_5_FOLD'])

# What sequence and functional features characterize the domain populations?

Sequence logos and functional annotation summaries highlight key functional motifs across the five structural domains.

## Sequence Logos

* **Domains 1 & 2 (SH3-like):** Low information content overall, reflecting broad sequence divergence across the fold.
* **Domains 3 & 4:** Characterized by highly conserved cysteines across the structural alignment.
* **Domain 5:** Features a distinct $\text{HxD}\dots\text{H}$ active-site motif.

In [ ]:
## Sequence logos use zero-based plot positions.
for view_name in ["dom_1", "dom_2", "dom_3", "dom_4"]:
    left, right = project.provenance[view_name].parameters["domain_ranges"][0]
    project.viz.plot_msa(f"{view_name}_FOLD", left, right)
# Show a smaller region of domain 5; motif/PDB residue parameters below remain one-based.
project.viz.plot_msa("dom_5_FOLD", 1499, 1660)


# Is there a conserved active/binding site in your structure?
*Map the conserved residues in the 3-D structure*

The sequence logos above show conserved cysteines and histidines, along with a putative catalytic signature. Here, we examine side-chain geometry in the Query structure for candidate metal-binding sites and highlight the corresponding residues in 3D.


In [ ]:
# Check metal binding sites in Query. Requires full coordinates.
results = daliscope.analysis.CH_cluster.evaluate_cys_his_sites(project.query_pdb_str)
daliscope.analysis.CH_cluster.print_report(results)

In [ ]:
# Positions identified as metal-binding or active site candidates
important_residues = [1523, 1525, 1652, 867, 877, 880, 900, 962, 1075, 1087, 1089, 994, 997, 1016, 1019, 1008, 1011, 1045, 1051]

# Highlight candidate metal binding residues and active site
prop = {} 
for i in important_residues[0:3]: prop[i] = 5   # domain 5 active site
for i in important_residues[3:7]: prop[i] = 4   # metal binding candidate
for i in important_residues[7:11]: prop[i] = 3  # metal binding candidate
for i in important_residues[11:15]: prop[i] = 2 # metal binding candidate
for i in important_residues[15:19]: prop[i] = 1  # metal binding candidate

daliscope.optics.py3Dmol_viewer.show_structure(
                    project.query_pdb_str,
                    important_residues,
                    property_dict=prop,
                    width=800,
                    height=600,
                    backbone_radius=0.18,
                    ca_radius=0.28,
                    highlight_scale=1.8)

The AlphaFold model places cysteine and histidine side chains in four spatial clusters with C4 or C3H composition, consistent with possible metal-binding sites. No metal ion is modeled here, so the geometry check identifies candidate ligand arrangements rather than metal–ligand bond lengths or confirmed binding sites. The highlighted HxD...H residues in Domain 5 mark a putative active site.


# Which proteins have the active site's signature motif?

We search the selected `dom_5_FOLD` population for HxD...H-related patterns at positions anchored to the structural alignment. The search identifies targets that pass the Domain 5 structural filter and have the specified residues at corresponding aligned positions. The following cell compares several pattern definitions and reports matches by Pfam assignment.


In [ ]:
import daliscope.optics.histograms

# define search population. We use a domain population for accurate Pfam annotations.
view_name = 'dom_5_FOLD'
parent_df = project.views[view_name]

daliscope.optics.histograms.set_motif(parent_df, [1523, 1525, 1652], k=4, verbose=True)

# Define all 3 patterns in the list
patterns = [
    [(1523, ["H"]), (1525, ["D"]), (1652, ["H"])],
    [(1523, ["H"]), (1525, ["D", "H"]), (1652, ["H"])],
    [(1523, ["H"]), (1652, ["H"])],
]

view = project.views[view_name]

# Loop over each pattern one by one
for i, pattern in enumerate(patterns, start=1):
    show_heading(f"Pattern {pattern}")
    # Pass a single pattern wrapped in a list: [pattern]
    hits = daliscope.optics.histograms.find_pattern_targets(view, [pattern])
    
    print(f"\n[{view_name}] Pattern {i}: {len(hits)} matches")
    print(f"Pattern definition: {pattern}\n")

    for category in ['clan', 'pfam']:
        # plot penetrance per category for this pattern
        daliscope.optics.histograms.draw_diverging_percentage_plot(
            view,
            hits=hits,
            grouping_col=category,
            yscale='linear',
        )

> In the supplied analysis, HxD...H occurs among CUPIN-annotated and unassigned hits. Inspect the printed counts and family/clan penetrance plots for the current run.
>
> The second position varies in the motif enumeration. Relaxing that position increases the number of matches; assess recall against the current domain selection and annotations rather than reusing counts from an earlier run.


In [ ]:
# 1. Get hits for the specified pattern
hits = daliscope.optics.histograms.find_pattern_targets(view, [patterns[1]])

# 2. Filter for hits where pfam is 'Unassigned'
df = project.views[view_name]
unassigned_hits_df = df[
    (df['target_id'].isin(hits)) & 
    (df['pfam'] == 'Unassigned')
]

# 3. Verify sequence context
tmp = list(unassigned_hits_df['sequ_pileup'])
for row in tmp:
    print(row[1500:1540],' ... ',row[1620:1660])

# 4. Print target_id and description
print(f"\nFound {len(unassigned_hits_df)} hits with 'Unassigned' Pfam:\n")
with pd.option_context('display.max_colwidth', None):
    display(unassigned_hits_df[['target_id', 'z_score', 'description']]) #.to_string(index=False))

Here, `Unassigned` means that no Pfam family was assigned to the matched region. It does not mean the entire protein is functionally uncharacterized: some descriptions refer to dioxygenase or JmjC proteins. Inspect the listed target descriptions and sequence context before interpreting these hits.


# Conclusion

This worked example uses DaliScope to examine five visually selected structural regions of a multidomain Query protein and compare their DALI target populations with Pfam annotations.

* **Domain architecture:** The structural alignment and occupancy views guide the choice of five domain boundaries and reveal matched regions with limited Pfam annotation.
* **Domain relationships:** Domains 1 and 2 have an SH3-like fold. Cysteine-rich Domains 3 and 4 often co-occur with the CUPIN-associated Domain 5, but the former are not themselves CUPIN domains.
* **Candidate sites:** The Query model contains four cysteine/histidine side-chain clusters consistent with possible metal-binding sites. The geometry alone does not establish metal occupancy or a regulatory function.
* **Structurally anchored motif search:** The HxD...H-related patterns are evaluated at corresponding positions within the selected Domain 5 population, and the resulting hits can be compared with Pfam and descriptive annotations.

Together, these analyses show how structural alignment can complement sequence-based annotations and suggest targets for further validation.


In [ ]:
print(FULL_DF[['sequence_identity','target_length','clan_domains']].iloc[0]['clan_domains'])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_worked_example_1_cartoon_staggered_labels():
    fig, ax = plt.subplots(figsize=(15, 4.8), dpi=300)
    
    seq_length = project.query_length
    backbone_y = 2.0
    
    # Pfam annotations with explicit label offsets to prevent collision
    # y_offset specifies how far below the box the text should sit
    pfam_domains = [
        {'id': 'PF22989 (DUF7030)', 'start': 8, 'end': 80, 'color': '#3498DB', 'height': 0.5, 'y_offset': -0.45},
        {'id': 'PF22987 (Tudor)', 'start': 153, 'end': 221, 'color': '#1ABC9C', 'height': 0.5, 'y_offset': -0.85},  # Staggered lower
        {'id': 'PF02373 (JmjC)', 'start': 1562, 'end': 1667, 'color': '#E67E22', 'height': 0.5, 'y_offset': -0.45}
    ]
    
    daliscope_domains = [
        {'name': 'Dom 1', 'start': 1, 'end': 149},
        {'name': 'Dom 2', 'start': 150, 'end': 214},
        {'name': 'Dom 3', 'start': 840, 'end': 920},
        {'name': 'Dom 4', 'start': 921, 'end': 1100},
        {'name': 'Dom 5 (CUPIN/JmjC core)', 'start': 1343, 'end': 1722}
    ]
    
    clusters = [
        {'name': 'Cluster 1 (Cys4)', 'positions': [867, 877, 880, 900], 'color': '#D35400'},
        {'name': 'Cluster 2 (Cys3His1)', 'positions': [962, 1075, 1087, 1089], 'color': '#8E44AD'},
        {'name': 'Cluster 3 (Cys4)', 'positions': [994, 997, 1016, 1019], 'color': '#27AE60'},
        {'name': 'Cluster 4 (Cys3His1)', 'positions': [1008, 1011, 1045, 1051], 'color': '#2980B9'}
    ]
    
    motif_positions = [('H1523', 1523), ('D1525', 1525), ('H1652', 1652)]

    # 1. Sequence Backbone Line
    ax.plot([1, seq_length], [backbone_y, backbone_y], color='#2C3E50', lw=3.5, zorder=1)
    ax.text(1, backbone_y + 0.70, '1', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.text(seq_length, backbone_y + 0.70, f'{seq_length} aa', ha='center', va='bottom', fontsize=12, fontweight='bold')

    # 2. Pfam Domains (Filled boxes centered on backbone; labels staggered below)
    for pfam in pfam_domains:
        width = pfam['end'] - pfam['start']
        h = pfam['height']
        center_x = pfam['start'] + width / 2
        box_bottom = backbone_y - h / 2
        
        rect = patches.Rectangle((pfam['start'], box_bottom), width, h,
                                 linewidth=1, edgecolor='#2C3E50', facecolor=pfam['color'], alpha=0.85, zorder=2)
        ax.add_patch(rect)
        
        text_y = backbone_y + pfam['y_offset']
        
        # If staggered lower, draw a subtle connector line from the box down to the label
        if pfam['y_offset'] < -0.5:
            ax.plot([center_x, center_x], [box_bottom, text_y + 0.05],
                    color='#7F8C8D', linestyle=':', linewidth=1.2, zorder=1)
            
        ax.text(center_x, text_y, pfam['id'],
                ha='center', va='top', fontsize=11, fontweight='bold', color='#1A252C')

    # 3. DaliScope Refined Domains (Labels above)
    for dom in daliscope_domains:
        width = dom['end'] - dom['start']
        h = 0.85
        rect = patches.Rectangle((dom['start'], backbone_y - h/2), width, h,
                                 linewidth=1.8, edgecolor='#C0392B', linestyle='--', facecolor='none', zorder=3)
        ax.add_patch(rect)
        ax.text(dom['start'] + width/2, backbone_y + 0.55, dom['name'],
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#C0392B')

    # 4. Large Colored Dots for Metal-Binding Cluster Residues
    for c in clusters:
        for pos in c['positions']:
            ax.scatter(pos, backbone_y, color=c['color'], edgecolor='#1A252C', linewidth=0.8, s=110, zorder=5)

    # 5. Motif Diamonds on Sequence Backbone
    for label, pos in motif_positions:
        ax.scatter(pos, backbone_y, color='#F1C40F', edgecolor='#7D6608', linewidth=1.0, s=110, marker='D', zorder=5)

    # 6. Callout Bracket / Zoom Inset
    ax.annotate('HxD...H Motif\n(H1523, D1525, H1652)', xy=(1524, backbone_y + 0.15), xytext=(1524, backbone_y + 1.35),
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#7D6608',
                arrowprops=dict(arrowstyle='->', color='#B7950B', lw=1.2, connectionstyle="arc3,rad=0"))

    # Legend
    legend_elements = [
        patches.Patch(facecolor='#3498DB', edgecolor='#2C3E50', alpha=0.85, label='Pfam Annotations'),
        patches.Patch(facecolor='none', edgecolor='#C0392B', linestyle='--', linewidth=1.5, label='DaliScope Domains'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#D35400', markeredgecolor='#1A252C', markersize=10, label='Cluster 1 (Cys4)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#8E44AD', markeredgecolor='#1A252C', markersize=10, label='Cluster 2 (Cys3His1)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#27AE60', markeredgecolor='#1A252C', markersize=10, label='Cluster 3 (Cys4)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#2980B9', markeredgecolor='#1A252C', markersize=10, label='Cluster 4 (Cys3His1)'),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='#F1C40F', markeredgecolor='#7D6608', markersize=10, label='HxD...H Motif (Dom 5)')
    ]
    
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.40), ncol=4, frameon=False, fontsize=11)

    ax.set_xlim(-40, seq_length + 60)
    ax.set_ylim(-0.2, 4.0)
    ax.axis('off')

    plt.tight_layout()
    return fig, ax

fig, ax = draw_worked_example_1_cartoon_staggered_labels()
plt.show()